In [1]:
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
import time
import requests  # For sending HTTP alerts
from datetime import datetime
import json

In [2]:
from PytorchWildlife.models import detection as pw_detection

detection_model = pw_detection.MegaDetectorV6(
    pretrained=True,
    version="MDV6-yolov10-c" 
)

# Run detection
img_path2 = "C:/Users/yrosh/Downloads/zebra.jpg"
img_path = "C:/Users/yrosh/Downloads/zebra.jpg"
detection_result = detection_model.single_image_detection(img_path)
detection_result

Ultralytics 8.4.40  Python-3.12.10 torch-2.11.0+cpu CPU (11th Gen Intel Core i5-11320H @ 3.20GHz)
YOLOv10n summary (fused): 101 layers, 2,265,753 parameters, 0 gradients, 6.5 GFLOPs

0: 1280x1280 3 animals, 1306.0ms
Speed: 24.6ms preprocess, 1306.0ms inference, 12.7ms postprocess per image at shape (1, 3, 1280, 1280)


{'img_id': 'C:/Users/yrosh/Downloads/zebra.jpg',
 'detections': Detections(xyxy=array([[      402.8,      72.506,      1001.1,      667.88],
        [     664.22,      234.28,      1254.2,      844.21],
        [     544.39,      232.28,      1076.5,       841.7]], dtype=float32), mask=None, confidence=array([    0.67021,     0.62807,     0.22553], dtype=float32), class_id=array([0, 0, 0]), tracker_id=None, data={}),
 'labels': ['animal 0.67', 'animal 0.63', 'animal 0.23'],
 'normalized_coords': [[np.float32(0.31468597),
   np.float32(0.085001364),
   np.float32(0.7821322),
   np.float32(0.7829729)],
  [np.float32(0.518925),
   np.float32(0.27465445),
   np.float32(0.9798813),
   np.float32(0.9896987)],
  [np.float32(0.4253057),
   np.float32(0.27231494),
   np.float32(0.8410465),
   np.float32(0.9867549)]]}

In [3]:
# Load  trained model and class names
model = tf.keras.models.load_model('./wildlife_classifier_class_15_acc_93.keras')

with open('./class_labels_class_15_acc_93.json', 'r') as f:
    class_names = json.load(f)

class_names

['Impala',
 'Indian_Elephant',
 'brown_bear',
 'cheetah',
 'coyote_indian_jackal',
 'gazelle',
 'jaguar',
 'langur',
 'leopard',
 'lion',
 'macaque',
 'puma',
 'sloth_bear',
 'tiger',
 'timber_wolf']

In [4]:
def classify_image_cnn(cropped_img, confidence_threshold=0.3):
    
    if cropped_img is None or cropped_img.size == 0:
        return "Invalid_Image", 0.0
    
    try:
        # Preprocessing
        img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (160, 160))           
        img = img / 255.0
        img = np.expand_dims(img, axis=0)
        
        # Predict
        predictions = model.predict(img, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = float(predictions[0][class_idx])
        
        species = class_names[class_idx]
        
        if confidence < confidence_threshold:
            return "Unknown", confidence
            
        return species, confidence
    
    except Exception as e:
        print(f"Classification error: {e}")
        return "Error", 0.0
    
    
image = cv2.imread("C:/Users/yrosh/Downloads/chita.jpg")
classify_image_cnn(image)

('cheetah', 0.6697331666946411)

In [5]:
def detect_and_classify(image_path, det_conf_threshold=0.5, cls_conf_threshold=0.3, show=True):
    
    start_time = time.time()

    #for images from internet
    # resp = requests.get(image_path)
    # img_array = np.asarray(bytearray(resp.content), dtype=np.uint8)
    # img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Could not load image")
        return None
    
    # Run detection
    detections = detection_model.single_image_detection(img)
    
    result_img = img.copy()
    results = []
    
    for box, conf, class_id in zip(detections['detections'].xyxy, detections['detections'].confidence, detections['detections'].class_id):
        if(class_id != 0):
            continue
        if conf < det_conf_threshold:
            continue
            
        x_min, y_min, x_max, y_max = map(int, box)
        
        # Crop detected animal
        cropped = img[y_min:y_max, x_min:x_max]
        
        # Classify with your CNN
        species, species_conf = classify_image_cnn(cropped, cls_conf_threshold)
        
        results.append({
            'species': species,
            'det_conf': float(conf),
            'cls_conf': species_conf,
            'bbox': [x_min, y_min, x_max, y_max]
        })
        
        # Draw on image
        color = (0, 255, 0) if species != "Unknown" else (0, 165, 255)
        cv2.rectangle(result_img, (x_min, y_min), (x_max, y_max), color, 2)
        
        label = f"{species} {species_conf:.2f}"
        cv2.putText(result_img, label, (x_min, y_min - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    
    total_time = time.time() - start_time
    
    # Print results
    print(f"\nDetection + Classification completed in {total_time:.2f} seconds")
    print(f"Detected {len(results)} animal(s)\n")
    
    for r in results:
        print(f"→ {r['species']:15} | Det_con: {r['det_conf']:.3f} | Classi_con: {r['cls_conf']:.3f}")
    
    # Show image
    if show:
        cv2.imshow("Wildlife Detection + Classification", result_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    
    
    return results


if __name__ == "__main__":
    
    # test_image = "./images/test5.jpg"   
    test_image = "./images/test4.jpg"   
    # test_image = "./images/test3.jpg"   
    # test_image = "./images/test2.jpg"   
    # test_image = "./images/test1.jpg"   
    detect_and_classify(test_image, det_conf_threshold=0.2, cls_conf_threshold=0.1)


0: 1280x1280 5 animals, 614.1ms
Speed: 18.5ms preprocess, 614.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

Detection + Classification completed in 1.32 seconds
Detected 5 animal(s)

→ tiger           | Det_con: 0.940 | Classi_con: 1.000
→ lion            | Det_con: 0.918 | Classi_con: 1.000
→ cheetah         | Det_con: 0.897 | Classi_con: 0.558
→ leopard         | Det_con: 0.786 | Classi_con: 0.810
→ jaguar          | Det_con: 0.745 | Classi_con: 0.709


KeyboardInterrupt: 

In [ ]:
import time
import threading
import requests
import cv2
import base64
from collections import deque
from datetime import datetime


class EdgeAlertManager:
    def __init__(self, camera_id, zone, location):
        self.camera_id = camera_id
        self.zone = zone
        self.location = location
        
        # Buffer to keep track of species detected in the last 5 sampled frames
        self.frame_buffer = deque(maxlen=5) 
        
        # Dictionary to track when an alert was last sent for a species { 'leopard': 1692461000.0 }
        self.last_alert_times = {} 
        
        self.COOLDOWN_SECONDS = 300 # 5 minutes
        self.API_URL = " https://webhook.site/95612668-05fc-40a9-9a95-4f6254483f63"

        # Define priority levels
        self.SEVERITY_MAP = {
            'leopard': 'CRITICAL',
            'tiger': 'CRITICAL',
            'sloth_bear': 'CRITICAL',
            'jaguar': 'CRITICAL',
            'cheetah': 'CRITICAL',
            'lion': 'HIGH',
            'puma': 'HIGH',
            'brown_bear': 'MEDIUM',
            'timber_wolf': 'MEDIUM',
            'coyote_indian_jackal': 'MEDIUM',
            'langur': 'LOW',
            'macaque': 'LOW',
        }

    def process_frame_detections(self, current_frame_species, best_crops_dict, max_conf_dict):
        self.frame_buffer.append(current_frame_species)
        
        # Flatten the buffer to count occurrences of each species in the last 5 frames
        species_counts = {}
        for frame_set in self.frame_buffer:
            for sp in frame_set:
                species_counts[sp] = species_counts.get(sp, 0) + 1
                
        # CRITICAL FIX: Only iterate over species detected in the CURRENT frame
        # This guarantees they exist in max_conf_dict and best_crops_dict
        for species in current_frame_species:
            if species_counts.get(species, 0) >= 2: # Note: Change back to 3 for production
                
                # Extract the specific confidence and image for this species
                conf = max_conf_dict[species]
                img = best_crops_dict[species]
                
                self._attempt_alert(species, conf, img)

    def _attempt_alert(self, species, confidence, frame_image):
        current_time = time.time()
        last_time = self.last_alert_times.get(species, 0)
        
        # Check Debounce / Cooldown
        if (current_time - last_time) < self.COOLDOWN_SECONDS:
            return # Skip, still in cooldown
            
        # Update cooldown timestamp immediately so we don't fire multiple threads
        self.last_alert_times[species] = current_time
        severity = self.SEVERITY_MAP.get(species, 'WARNING')
        
        # Dispatch HTTP request in a separate thread! (This fixes the blocking issue)
        threading.Thread(
            target=self._send_payload_async, 
            args=(species, confidence, severity, frame_image),
            daemon=True
        ).start()

    def _send_payload_async(self, species, confidence, severity, frame_image):
        try:
            # Optional: Convert image to base64 to send directly to Node.js
            _, buffer = cv2.imencode('.jpg', frame_image)
            img_b64 = base64.b64encode(buffer).decode('utf-8')
            
            payload = {
              "camera_id": self.camera_id,
              "zone": self.zone,
              "location": self.location,
              "species": species,
              "confidence": round(float(confidence), 3),
              "severity": severity,
              "timestamp": datetime.utcnow().isoformat() + "Z",
              "image_base64": img_b64 # Node.js will upload this to Cloudinary
            }

            
            response = requests.post(self.API_URL, json=payload, timeout=10)
            print(f"[ALERT SENT] {species} - Status: {response.status_code}")
        except Exception as e:
            print(f"[ALERT FAILED] {e}")

In [31]:
import cv2

# Notice we added alert_manager as a parameter here
def process_video(video_path, alert_manager, det_conf_threshold=0.5, cls_conf_threshold=0.3, frame_skip=4):
    
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    
    print(f"Processing video: {video_path}")
    
    while cap.isOpened():
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        success, frame = cap.read()
        
        if not success:
            break

        small_frame = cv2.resize(frame, (640, 360))
        detections = detection_model.single_image_detection(small_frame)
        
        current_species = set()
        max_conf_dict = {}
        best_crops_dict = {} # Changed from a single variable to a dictionary

        for box, conf, cls_id in zip(detections['detections'].xyxy, 
                                     detections['detections'].confidence, 
                                     detections['detections'].class_id):
            
            if cls_id == 0 and conf > det_conf_threshold:  
                x_min, y_min, x_max, y_max = map(int, box)
                cropped_img = small_frame[y_min:y_max, x_min:x_max]
                
                if cropped_img.size == 0:
                    continue
                
                species, species_conf = classify_image_cnn(cropped_img, cls_conf_threshold)
                
                # Only process if it meets the high threshold for alerts
                if species_conf > 0.7:
                    current_species.add(species)
                    
                    # If this is the first time seeing this species in this frame, 
                    # or if the confidence is higher than a previous detection in the same frame
                    if species_conf > max_conf_dict.get(species, 0):
                        max_conf_dict[species] = species_conf
                        best_crops_dict[species] = cropped_img # Save specific crop

                # Draw bounding box and label
                color = (0, 255, 0) if species != "Unknown" else (0, 165, 255)
                cv2.rectangle(small_frame, (x_min, y_min), (x_max, y_max), color, 2)
                label = f"{species[:15]} {species_conf:.2f}"
                cv2.putText(small_frame, label, (x_min, y_min-8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow("Wildlife Detection", small_frame)

        # Trigger alert manager
        if current_species:
            # We now pass the dictionary of crops so the manager can pick the right one
            alert_manager.process_frame_detections(current_species, best_crops_dict, max_conf_dict)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        frame_id += frame_skip + 1

    cap.release()
    cv2.destroyAllWindows()
    print("Video processing completed.")


if __name__ == "__main__":
    video_path = "C:/Users/yrosh/Downloads/vidssave.com Cheetah Sneaks Up to Attack Sleeping Man 480P.mp4"   
    
    # 1. Initialize your alert manager first
    my_alert_manager = EdgeAlertManager(
        camera_id="CAM_Aroli_Gate_04", 
        zone="ZoneA",
        location={"lat": 19.148, "lng": 72.932, "area_name": "Sector 4, Near Forest Boundary"}
    )
    
    # 2. Pass it into the function
    process_video(
        video_path=video_path,
        alert_manager=my_alert_manager, 
        det_conf_threshold=0.5, # Changed back from 0 so it actually filters
        cls_conf_threshold=0.3, 
        frame_skip=8         
    )

Processing video: C:/Users/yrosh/Downloads/vidssave.com Cheetah Sneaks Up to Attack Sleeping Man 480P.mp4

0: 1280x1280 1 vehicle, 475.5ms
Speed: 17.8ms preprocess, 475.5ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 668.8ms
Speed: 20.9ms preprocess, 668.8ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 526.3ms
Speed: 19.5ms preprocess, 526.3ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 479.6ms
Speed: 12.9ms preprocess, 479.6ms inference, 2.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 839.2ms
Speed: 27.3ms preprocess, 839.2ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 582.8ms
Speed: 28.6ms preprocess, 582.8ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 vehicle, 660.6ms
Speed: 30.9ms preprocess, 660.6ms inference, 0.


0: 1280x1280 1 animal, 812.9ms
Speed: 28.5ms preprocess, 812.9ms inference, 1.7ms postprocess per image at shape (1, 3, 1280, 1280)
[ALERT SENT] cheetah - Status: 200

0: 1280x1280 1 animal, 469.0ms
Speed: 22.2ms preprocess, 469.0ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 486.5ms
Speed: 18.8ms preprocess, 486.5ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 vehicle, 570.5ms
Speed: 26.3ms preprocess, 570.5ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 vehicle, 443.9ms
Speed: 14.3ms preprocess, 443.9ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 442.2ms
Speed: 13.7ms preprocess, 442.2ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 2 vehicles, 439.0ms
Speed: 13.3ms preprocess, 439.0ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1